In [4]:
import pandas as pd
import random
import psycopg2
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# ======================
# CONFIG (DÙNG DIRECT DB)
# ======================
DB_CONFIG = {
    "host": "aws-1-ap-northeast-2.pooler.supabase.com",  # 🔥 đổi lại
    "port": 6543,                          # 🔥 không dùng 6543 nữa
    "database": "postgres",
    "user": "postgres.ypkwqsbsunlvpoqdadbq",
    "password": "5$eAK8EV4S+gsKj",
    "sslmode": "require"
}

CSV_PATH = "./database/foods.csv"
RATING_PATH = "./database/ratings.csv"

GROUP_SIZE = 5
BATCH_SIZE = 50

# ======================
# CONNECT DB
# ======================
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

print("Connected to DB 🚀")

# ======================
# MODEL
# ======================
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(CSV_PATH)
df = df.fillna("")

# nếu chưa có food_id → tạo
if "food_id" not in df.columns:
    df["food_id"] = df.index

# ======================
# HELPER
# ======================
def generate_location():
    return (
        10.75 + random.uniform(-0.05, 0.05),
        106.67 + random.uniform(-0.05, 0.05)
    )

def clean_tags(tags):
    return [t.strip() for t in str(tags).split(",") if t.strip()]

def build_text(rows):
    return " ".join(
        f"{r['dish_name']} {r['description']} {r.get('dish_tags','')}"
        for _, r in rows.iterrows()
    )

# ======================
# INSERT DATA
# ======================
counter = 0

try:
    for i in tqdm(range(0, len(df), GROUP_SIZE)):

        group = df.iloc[i:i+GROUP_SIZE]

        if len(group) < GROUP_SIZE:
            break

        # -------- RESTAURANT --------
        lat, lon = generate_location()

        cur.execute("""
            INSERT INTO restaurants (name, price_level, latitude, longitude)
            VALUES (%s, %s, %s, %s)
            RETURNING id
        """, (
            f"Quán {group.iloc[0]['dish_name']}",
            random.randint(1, 3),
            lat,
            lon
        ))

        restaurant_id = cur.fetchone()[0]

        # -------- MENUS --------
        for _, row in group.iterrows():
            cur.execute("""
                INSERT INTO menus (id, restaurant_id, dish_name, description, tags)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (id) DO NOTHING
            """, (
                int(row["food_id"]),
                restaurant_id,
                row.get("dish_name", ""),
                row.get("description", ""),
                clean_tags(row.get("dish_tags", ""))
            ))

        # -------- EMBEDDING --------
        content = build_text(group)
        embedding = model.encode(content).tolist()

        cur.execute("""
            INSERT INTO restaurant_embeddings (restaurant_id, content, embedding)
            VALUES (%s, %s, %s)
        """, (
            restaurant_id,
            content,
            embedding
        ))

        counter += 1

        # 🔥 commit theo batch
        if counter % BATCH_SIZE == 0:
            conn.commit()
            print(f"✅ committed {counter} restaurants")

    conn.commit()
    print("Insert restaurants DONE 🚀")

except Exception as e:
    conn.rollback()
    print("❌ ERROR:", e)

# ======================
# RATING PIPELINE
# ======================
try:
    ratings_df = pd.read_csv(RATING_PATH)

    # load mapping từ menus
    cur.execute("SELECT id, restaurant_id FROM menus")
    mapping = dict(cur.fetchall())

    restaurant_ratings = {}

    for _, row in ratings_df.iterrows():
        user = int(row["userId"])
        food = int(row["foodId"])
        rating = float(row["rating"])

        if rating == 0 or food not in mapping:
            continue

        rest = mapping[food]
        key = (user, rest)

        restaurant_ratings.setdefault(key, []).append(rating)

    # insert rating
    for (user, rest), ratings in restaurant_ratings.items():
        avg_rating = sum(ratings) / len(ratings)

        cur.execute("""
            INSERT INTO user_ratings (user_id, restaurant_id, rating)
            VALUES (%s, %s, %s)
        """, (user, rest, avg_rating))

    conn.commit()
    print("Rating DONE ⭐")

except Exception as e:
    conn.rollback()
    print("❌ Rating ERROR:", e)

cur.close()
conn.close()

Connected to DB 🚀


  6%|▋         | 50/800 [00:50<12:36,  1.01s/it]

✅ committed 50 restaurants


 12%|█▎        | 100/800 [01:36<10:51,  1.08it/s]

✅ committed 100 restaurants


...

 94%|█████████▍| 750/800 [13:04<01:00,  1.21s/it]

✅ committed 750 restaurants


100%|██████████| 800/800 [14:13<00:00,  1.07s/it]

✅ committed 800 restaurants
Insert restaurants DONE 🚀


KeyboardInterrupt: 

In [6]:
import pandas as pd
import psycopg2
from tqdm import tqdm

# ======================
# CONNECT DB
# ======================
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

print("Connected to DB 🚀")

try:
    ratings_df = pd.read_csv(RATING_PATH)

    # load mapping
    cur.execute("SELECT id, restaurant_id FROM menus")
    mapping = dict(cur.fetchall())

    print(f"📊 Total ratings: {len(ratings_df)}")
    print(f"📊 Total menus loaded: {len(mapping)}")

    restaurant_ratings = {}

    # ======================
    # PROCESS RATING (có progress)
    # ======================
    for _, row in tqdm(ratings_df.iterrows(), total=len(ratings_df), desc="Processing ratings"):

        user = int(row["userId"])
        food = int(row["foodId"])
        rating = float(row["rating"])

        if rating == 0 or food not in mapping:
            continue

        rest = mapping[food]
        key = (user, rest)

        restaurant_ratings.setdefault(key, []).append(rating)

    print(f"✅ Aggregated pairs: {len(restaurant_ratings)}")

    # ======================
    # INSERT RATING (có progress)
    # ======================
    inserted = 0

    for (user, rest), ratings in tqdm(restaurant_ratings.items(), desc="Inserting ratings"):

        avg_rating = sum(ratings) / len(ratings)

        cur.execute("""
            INSERT INTO user_ratings (user_id, restaurant_id, rating)
            VALUES (%s, %s, %s)
        """, (user, rest, avg_rating))

        inserted += 1

        # commit mỗi 100 record để an toàn
        if inserted % 100 == 0:
            conn.commit()

    conn.commit()

    print("🎉 DONE")
    print(f"📌 Total inserted: {inserted}")

except Exception as e:
    conn.rollback()
    print("❌ Rating ERROR:", e)

cur.close()
conn.close()

Connected to DB 🚀
📊 Total ratings: 182678
📊 Total menus loaded: 4000


Processing ratings: 100%|██████████| 182678/182678 [00:07<00:00, 24627.24it/s]


✅ Aggregated pairs: 75078


Inserting ratings:  31%|███       | 23196/75078 [1:07:05<2:30:02,  5.76it/s]    


InterfaceError: connection already closed